# MoodMate: Random Forest Feature Experiments

This notebook is a **collaborative MoodMate project** experiment comparing Random Forest emotion classification with bag-of-words counts and TF-IDF features. The six labels are sadness (0), joy (1), love (2), anger (3), fear (4), and surprise (5).


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score

from pathlib import Path

data_dir = Path("data") if Path("data/training.csv").is_file() else Path(".")
required_files = ("training.csv", "validation.csv", "test.csv")
missing = [name for name in required_files if not (data_dir / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} in {data_dir.resolve()}. "
        "Place the original CSV files in a data/ folder or beside this notebook."
    )

train_df = pd.read_csv(data_dir / "training.csv")
val_df = pd.read_csv(data_dir / "validation.csv")
test_df = pd.read_csv(data_dir / "test.csv")
for split_name, frame in (("training", train_df), ("validation", val_df), ("test", test_df)):
    if not {"text", "label"}.issubset(frame.columns):
        raise ValueError(f"{split_name}.csv must contain 'text' and 'label' columns")
    if frame[["text", "label"]].isna().any().any():
        raise ValueError(f"{split_name}.csv contains missing text or labels")

LABEL_NAMES = {0: "sadness", 1: "joy", 2: "love", 3: "anger", 4: "fear", 5: "surprise"}

## Training label distribution

In [ ]:
counts = train_df["label"].value_counts().sort_index()
ax = sns.barplot(x=[LABEL_NAMES[i] for i in counts.index], y=counts.values, color="steelblue")
for i, count in enumerate(counts.values):
    ax.text(i, count, str(count), ha="center", va="bottom", fontsize=9)
ax.set(title="Training label distribution", xlabel="Emotion", ylabel="Messages")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Shared train, validation, and test splits

The original project provides predefined splits. Both experiments use the same splits without creating a second random partition.

In [ ]:
X_train, y_train = train_df["text"], train_df["label"]
X_val, y_val = val_df["text"], val_df["label"]
X_test, y_test = test_df["text"], test_df["label"]

## Random Forest with CountVectorizer

Bag-of-words features represent token counts. Fit the vocabulary only on training data and retain the original 100-tree model and random seed.

In [ ]:
count_vect = CountVectorizer()
X_train_count = count_vect.fit_transform(X_train)
X_val_count = count_vect.transform(X_val)
X_test_count = count_vect.transform(X_test)

rf_count = RandomForestClassifier(n_estimators=100, random_state=42)
rf_count.fit(X_train_count, y_train)

count_val_accuracy = accuracy_score(y_val, rf_count.predict(X_val_count))
count_test_accuracy = accuracy_score(y_test, rf_count.predict(X_test_count))
print(f"Validation Accuracy of Random Forest Classifier: {count_val_accuracy:.2f}")
print(f"Test Accuracy of Random Forest Classifier: {count_test_accuracy:.2f}")

Validation Accuracy of Random Forest Classifier: 0.88
Test Accuracy of Random Forest Classifier: 0.87


**Original saved results:** 0.88 validation accuracy and 0.87 test accuracy (rounded to two decimal places).

## Random Forest with TF-IDF

Repeat the same model configuration with TF-IDF features. Unlike the main MoodMate notebook, this original experiment used default `TfidfVectorizer()` settings, **without** English stop-word removal. The results are therefore not directly controlled comparisons against the main notebook.

In [ ]:
tfidf_vect = TfidfVectorizer()
X_train_tfidf = tfidf_vect.fit_transform(X_train)
X_val_tfidf = tfidf_vect.transform(X_val)
X_test_tfidf = tfidf_vect.transform(X_test)

rf_tfidf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_tfidf.fit(X_train_tfidf, y_train)

tfidf_val_accuracy = accuracy_score(y_val, rf_tfidf.predict(X_val_tfidf))
tfidf_test_accuracy = accuracy_score(y_test, rf_tfidf.predict(X_test_tfidf))
print(f"Validation Accuracy of Random Forest Classifier: {tfidf_val_accuracy:.2f}")
print(f"Test Accuracy of Random Forest Classifier: {tfidf_test_accuracy:.2f}")

Validation Accuracy of Random Forest Classifier: 0.87
Test Accuracy of Random Forest Classifier: 0.86


**Original saved results:** 0.87 validation accuracy and 0.86 test accuracy (rounded to two decimal places).

## Most frequent training tokens

Display the most common terms in the **CountVectorizer training matrix**. This repairs the original cell, which referenced undefined `X` and `vectorizer` variables. Frequencies are token counts, not TF-IDF weights.

In [ ]:
word_counts = X_train_count.sum(axis=0).A1
word_freq_df = (
    pd.DataFrame({"word": count_vect.get_feature_names_out(), "frequency": word_counts})
    .sort_values("frequency", ascending=False)
    .head(20)
    .reset_index(drop=True)
)
word_freq_df

## Interpretation and limitations

The saved results favor CountVectorizer over TF-IDF for this Random Forest setup, but the differences are small and only reported to two decimal places.